In [1]:
# ═══════════════════════════════════════════════
# STEP 1: Install
# ═══════════════════════════════════════════════
!pip install ultralytics opencv-python-headless -q

# ═══════════════════════════════════════════════
# STEP 2: Imports
# ═══════════════════════════════════════════════
import cv2, numpy as np, time, os
from dataclasses import dataclass, field
from typing import List, Optional, Tuple, Dict
from ultralytics import YOLO

# ═══════════════════════════════════════════════
# STEP 3: Upload Video
# ═══════════════════════════════════════════════
os.makedirs("/content/data", exist_ok=True)
os.makedirs("/content/output/analytics", exist_ok=True)

from google.colab import files
print("📁 Apni downloaded football video select karo...")
uploaded = files.upload()
for f in uploaded.keys():
    os.rename(f, "/content/data/sports_video.mp4")
    size = os.path.getsize("/content/data/sports_video.mp4")/1024/1024
    print(f"✅ Video ready! {size:.1f} MB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 67.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
📁 Apni downloaded football video select karo...


Saving 26.mp4 to 26.mp4
✅ Video ready! 13.1 MB


In [2]:
# ═══════════════════════════════════════════════
# STEP 4: Helper Functions
# ═══════════════════════════════════════════════

def id_to_color(tid):
    np.random.seed(tid * 17 + 31)
    hue = int(np.random.uniform(0, 180))
    sat = int(np.random.uniform(180, 255))
    bgr = cv2.cvtColor(np.uint8([[[hue,sat,240]]]), cv2.COLOR_HSV2BGR)[0][0]
    return (int(bgr[0]), int(bgr[1]), int(bgr[2]))

def annotate(frame, tid, conf, box, trajectory, color):
    x1,y1,x2,y2 = box
    cx,cy = (x1+x2)//2, (y1+y2)//2
    # Box
    cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
    # Label
    label = f"ID {tid} {conf:.0%}"
    (lw,lh),_ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
    cv2.rectangle(frame, (x1,y1-lh-6), (x1+lw+4,y1), color, -1)
    cv2.putText(frame, label, (x1+2,y1-3),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1, cv2.LINE_AA)
    # Trajectory
    if len(trajectory) > 1:
        pts = trajectory[-40:]
        for i in range(1, len(pts)):
            alpha = i/len(pts)
            tc = tuple(int(ch*alpha) for ch in color)
            cv2.line(frame, pts[i-1], pts[i], tc, 2, cv2.LINE_AA)
    return frame, cx, cy

def draw_hud(frame, fn, total, active, total_ids):
    ov = frame.copy()
    cv2.rectangle(ov, (0,0), (280,85), (10,10,10), -1)
    cv2.addWeighted(ov, 0.5, frame, 0.5, 0, frame)
    if total > 0:
        cv2.rectangle(frame, (1,81), (int(278*fn/total),85), (80,200,120), -1)
    info = [
        ("PREDUSK TRACKER",       (8,16), 0.44, (100,220,255)),
        (f"Frame {fn}/{total}",   (8,35), 0.42, (200,200,200)),
        (f"Active : {active}",    (8,53), 0.42, (200,200,200)),
        (f"Total IDs : {total_ids}",(8,71),0.42,(120,255,160)),
    ]
    for txt,pos,sc,col in info:
        cv2.putText(frame, txt, pos, cv2.FONT_HERSHEY_SIMPLEX, sc, col, 1, cv2.LINE_AA)
    return frame

print("✅ Functions ready!")

✅ Functions ready!


In [4]:
INPUT_PATH  = "/content/data/sports_video.mp4"
OUTPUT_PATH = "/content/output/tracked_output.mp4"

print("Loading YOLOv8...")
model = YOLO("yolov8m.pt")
print("Model ready!\n")

cap     = cv2.VideoCapture(INPUT_PATH)
src_fps = cap.get(cv2.CAP_PROP_FPS)
src_w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
src_h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fourcc  = cv2.VideoWriter_fourcc(*"mp4v")
writer  = cv2.VideoWriter(OUTPUT_PATH, fourcc, src_fps, (src_w, src_h))

trajectories = {}
colors = {}
global_ids = set()
frame_counts = []
frame_num = 0
t_start = time.time()
print(f"Starting! Total frames: {total}\n")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    results = model.track(frame, persist=True, conf=0.35, iou=0.5, classes=[0], tracker="bytetrack.yaml", verbose=False)
    active_ids = set()
    if results and results[0].boxes is not None:
        for box in results[0].boxes:
            if box.id is None:
                continue
            tid  = int(box.id.item())
            conf = float(box.conf.item())
            x1,y1,x2,y2 = box.xyxy[0].cpu().numpy().astype(int)
            if tid not in colors:
                colors[tid] = id_to_color(tid)
                trajectories[tid] = []
                global_ids.add(tid)
            trajectories[tid].append(((x1+x2)//2, (y1+y2)//2))
            if len(trajectories[tid]) > 500:
                trajectories[tid] = trajectories[tid][-500:]
            active_ids.add(tid)
            frame, cx, cy = annotate(frame, tid, conf, (x1,y1,x2,y2), trajectories[tid], colors[tid])
    frame = draw_hud(frame, frame_num, total, len(active_ids), len(global_ids))
    writer.write(frame)
    frame_counts.append(len(active_ids))
    if frame_num % 300 == 0:
        print(f"  {frame_num/total*100:.1f}% | frame {frame_num}/{total} | IDs: {len(global_ids)} | {time.time()-t_start:.0f}s")
    frame_num += 1

cap.release()
writer.release()
print(f"\nDONE! Frames: {frame_num} | Unique IDs: {len(global_ids)} | Time: {time.time()-t_start:.1f}s")
print(f"Output: {OUTPUT_PATH}")

Loading YOLOv8...
Model ready!

Starting! Total frames: 2851

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 170ms
Prepared 1 package in 20ms
Installed 1 package in 4ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

  0.0% | frame 0/2851 | IDs: 17 | 4s
  10.5% | frame 300/2851 | IDs: 63 | 13s
  21.0% | frame 600/2851 | IDs: 130 | 23s
  31.6% | frame 900/2851 | IDs: 168 | 32s
  42.1% | frame 1200/2851 | IDs: 187 | 41s
  52.6% | frame 1500/2851 | IDs: 243 | 50s
  63.1% | frame 1800/2851 | IDs: 307 | 60s
  73.7% | frame 2100/2851 | IDs: 385 | 71s
  84.2% | frame 2400/2851 | IDs: 481 | 81s
  94.7% | frame 2700/2851 | IDs: 520 | 90s

DONE! Frames: 2848 | Unique IDs: 547 | Time: 94.4s
Output: /content/output/tracked_output.mp4


In [5]:
from google.colab import files
print("📥 Downloading output video...")
files.download("/content/output/tracked_output.mp4")
print("✅ Done!")

📥 Downloading output video...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Done!
